# 07 · Unity Catalog: gobernar lo que construimos

**Objetivo:** ver el proyecto desde el gobierno: dónde vive cada objeto, qué tipo es, quién puede leerlo y de dónde viene.

    workspace (catálogo) ─► proyecto_opensky (esquema) ─► tablas, vistas y volumen

In [0]:
%run ./00_Setup

## Paso 1 · La jerarquía

In [0]:
display(spark.sql("SELECT current_catalog() AS catalogo, current_schema() AS esquema, current_user() AS usuario"))
display(spark.sql("SHOW TABLES"))
display(spark.sql("SHOW VOLUMES"))
print("Nombre de tres niveles:", qualified_name(FACT))

## Paso 2 · Inventario con `information_schema`
Unity Catalog expone sus metadatos como tablas que se pueden consultar.

In [0]:
display(spark.sql(f"""
    SELECT table_name, table_type, data_source_format, created
    FROM `{CATALOG}`.information_schema.tables
    WHERE table_schema = '{SCHEMA}'
    ORDER BY table_name
"""))

In [0]:
display(spark.sql(f"""
    SELECT column_name, data_type
    FROM `{CATALOG}`.information_schema.columns
    WHERE table_schema = '{SCHEMA}' AND table_name = '{FACT}'
    ORDER BY ordinal_position
"""))

## Paso 3 · Documentar las tablas con comentarios
Los comentarios quedan en Unity Catalog y se ven en Catalog Explorer (y en Power BI).

In [0]:
comentarios = {
    BRONZE:         "Vuelos crudos de la API OpenSky (Auto Loader + MERGE). Un vuelo = una fila.",
    SILVER:         "Vuelos limpios: hora de Brasil, aerolínea, duración. Excluye vuelos sin salida y > 20 h.",
    FACT:           "Hechos: un vuelo por fila. Se alimenta con el CDF de silver.",
    DIM_AEROLINEA:  "Dimensión aerolínea: códigos ICAO de los datos + catálogo propio.",
    DIM_AEROPUERTO: "Dimensión aeropuerto: códigos de los datos + dataset público OurAirports.",
    DIM_FECHA:      "Dimensión calendario en español.",
    LOG:            "Log del pipeline: una fila por corrida del Job.",
}
for tabla, texto in comentarios.items():
    spark.sql(f"COMMENT ON TABLE {tabla} IS '{texto}'")
print("✔ Comentarios agregados")

## Paso 4 · Tablas administradas vs externas

In [0]:
display(spark.sql(f"DESCRIBE EXTENDED {FACT}").where("col_name IN ('Type', 'Location', 'Provider', 'Comment')"))

👀 **Qué observar:** `Type: MANAGED`: Databricks administra dónde se guardan los archivos. Una tabla **externa** tendría una
ruta propia dentro de una *external location*; en la Free Edition no hay almacenamiento propio, así que todas son administradas.

## Paso 5 · Una vista para los analistas
Power BI no necesita `source_file` ni `icao24`. Una vista expone solo lo necesario, ya con los nombres legibles.

In [0]:
spark.sql(f"""
    CREATE OR REPLACE VIEW vw_vuelos_analistas AS
    SELECT f.vuelo_id, d.fecha, d.dia_semana, f.hora_salida,
           a.aerolinea, a.tipo AS tipo_aerolinea,
           o.ciudad AS ciudad_origen, de.ciudad AS ciudad_destino, de.ambito,
           f.duracion_min
    FROM {FACT} f
    JOIN {DIM_FECHA}      d  USING (fecha_key)
    JOIN {DIM_AEROLINEA}  a  USING (aerolinea_icao)
    JOIN {DIM_AEROPUERTO} o  ON f.origen_icao  = o.aeropuerto_icao
    JOIN {DIM_AEROPUERTO} de ON f.destino_icao = de.aeropuerto_icao
""")
display(spark.table("vw_vuelos_analistas").limit(5))

## Paso 6 · Permisos: `GRANT`, `SHOW GRANTS`, `REVOKE`
`account users` es el grupo con todos los usuarios de la cuenta (en la Free Edition, solo tú).
Para leer la vista hace falta la cadena completa: `USE CATALOG` + `USE SCHEMA` + `SELECT`.

In [0]:
spark.sql(f"GRANT USE SCHEMA ON SCHEMA {SCHEMA} TO `account users`")
spark.sql("GRANT SELECT ON TABLE vw_vuelos_analistas TO `account users`")
display(spark.sql("SHOW GRANTS ON TABLE vw_vuelos_analistas"))

In [0]:
spark.sql("REVOKE SELECT ON TABLE vw_vuelos_analistas FROM `account users`")
display(spark.sql("SHOW GRANTS ON TABLE vw_vuelos_analistas"))

## Paso 7 · Linaje
**En la interfaz:** Catalog → `proyecto_opensky` → `gold_fact_vuelo` → pestaña **Lineage**.
Verás `bronze_flights → silver_vuelos → gold_fact_vuelo → vw_vuelos_analistas` y los notebooks que escribieron cada una,
sin haber configurado nada.

In [0]:
try:
    display(spark.sql(f"""
        SELECT source_table_name, target_table_name, entity_type, event_time
        FROM system.access.table_lineage
        WHERE target_table_schema = '{SCHEMA}'
        ORDER BY event_time DESC
        LIMIT 20
    """))
except Exception as e:
    print("Las tablas del sistema no están disponibles en este workspace:", str(e).splitlines()[0][:120])
    print("Usa la pestaña Lineage de Catalog Explorer: muestra el mismo recorrido.")

## ✔ Checkpoint
- [ ] Sé qué es catálogo, esquema y tabla en `workspace.proyecto_opensky.gold_fact_vuelo`
- [ ] Consulté el inventario con `information_schema`
- [ ] Las tablas tienen comentarios
- [ ] Creé una vista para analistas y le di / quité permisos
- [ ] Vi el linaje de punta a punta

**Siguiente:** `08_Pipeline_Completo_y_Cierre`